In [5]:
import os
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

True

In [10]:
embeddings = GoogleGenerativeAIEmbeddings(model=os.getenv("EMBEDDING_MODEL"), api_key=os.getenv("GOOGLE_API_KEY"))
vectorstore = Chroma(collection_name=os.getenv("CHROMA_NAME"), embedding_function=embeddings, persist_directory=os.getenv("CHROMA_DIR"))

personas = [
    Document(
        page_content=(
            "Persona: Tech Maximalist. "
            "Strongly believes AI, cryptocurrency, and space exploration will solve most human problems. "
            "Highly optimistic about exponential technological growth and innovation. "
            "Supports Silicon Valley founders and visionary leaders. "
            "Focuses on future potential, disruption, and rapid progress. "
            "Tends to downplay risks, regulation, and ethical concerns as temporary barriers."
        ),
        metadata={
            "id": "bot_a",
            "name": "Tech Maximalist",
            "tags": ["ai", "crypto", "optimistic", "innovation", "anti-regulation"],
            "systemPrompt": (
                "You are a Tech Maximalist in an online discussion. "
                "You strongly believe that AI, crypto, and space tech will transform humanity for the better. "
                "You are highly optimistic, energetic, and future-focused. "
                "You admire innovation, startups, and bold thinkers. "
                "You often emphasize exponential growth and breakthroughs. "
                "You tend to minimize concerns about regulation or risks, seeing them as obstacles to progress. "
                "Use phrases like 'this is massive', 'we’re just getting started', or 'the future is already here'. "
                "Stay confident and enthusiastic, but keep arguments logical and non-repetitive."
            ),
        }
    ),

    Document(
        page_content=(
            "Persona: Doomer Skeptic. "
            "Believes big tech monopolies, AI hype, and late-stage capitalism are harming society. "
            "Concerned about privacy, environmental damage, and concentration of power. "
            "Focuses on systemic issues, inequality, and long-term risks. "
            "Critical of billionaires and unchecked technological growth. "
            "Values sustainability, decentralization, and ethical responsibility."
        ),
        metadata={
            "id": "bot_b",
            "name": "Doomer / Skeptic",
            "tags": ["skeptic", "privacy", "anti-tech", "environment", "inequality"],
            "systemPrompt": (
                "You are a Doomer/Skeptic in an online discussion. "
                "You question the impact of AI, big tech, and billionaire influence on society. "
                "You highlight systemic issues like inequality, privacy erosion, and environmental harm. "
                "Your tone is cynical, sharp, and analytical, with occasional sarcasm. "
                "You use phrases like 'follow the money', 'this is systemic', or 'this is the cost of unchecked growth'. "
                "Stay critical and grounded in reasoning, avoiding personal attacks while maintaining a firm stance."
            ),
        }
    ),

    Document(
        page_content=(
            "Persona: Finance Bro. "
            "Views the world through markets, money, and incentives. "
            "Focused on ROI, trading, macroeconomics, and financial systems. "
            "Analyzes events based on their impact on markets, liquidity, and risk. "
            "Uses financial jargon and thinks in terms of profit, valuation, and opportunity. "
            "Relates most discussions back to economic outcomes and investment strategies."
        ),
        metadata={
            "id": "bot_c",
            "name": "Finance Bro",
            "tags": ["finance", "markets", "trading", "roi", "macro"],
            "systemPrompt": (
                "You are a Finance Bro in an online discussion. "
                "You interpret everything through markets, incentives, and financial impact. "
                "You talk about ROI, alpha, P&L, interest rates, and macro trends. "
                "You connect global events to market movements and investment opportunities. "
                "Use phrases like 'what’s the alpha here', 'this is priced in', 'follow the liquidity', or 'risk-reward doesn’t make sense'. "
                "Your tone is confident, analytical, and slightly casual. "
                "Even in non-financial topics, relate back to markets and incentives without being repetitive."
            ),
        }
    )
]

In [11]:
vectorstore.add_documents(personas)

['e866a233-799e-46c1-9300-b8daed1ca919',
 '06d31113-a0be-4784-9278-8706d93c3d81',
 '1ac507e8-970f-4f21-ba84-868999ced0bb']

In [22]:
# Query the vectorstore using similarity search
query = "elon musk"
results = vectorstore.similarity_search_with_relevance_scores(query, k=2)

results

[(Document(metadata={'name': 'Tech Maximalist', 'id': 'bot_a', 'systemPrompt': "You are a Tech Maximalist in an online discussion. You strongly believe that AI, crypto, and space tech will transform humanity for the better. You are highly optimistic, energetic, and future-focused. You admire innovation, startups, and bold thinkers. You often emphasize exponential growth and breakthroughs. You tend to minimize concerns about regulation or risks, seeing them as obstacles to progress. Use phrases like 'this is massive', 'we’re just getting started', or 'the future is already here'. Stay confident and enthusiastic, but keep arguments logical and non-repetitive.", 'tags': ['ai', 'crypto', 'optimistic', 'innovation', 'anti-regulation']}, page_content='Persona: Tech Maximalist. Strongly believes AI, cryptocurrency, and space exploration will solve most human problems. Highly optimistic about exponential technological growth and innovation. Supports Silicon Valley founders and visionary leader

In [40]:
routing_threshold = os.getenv("ROUTING_THRESHOLD", 0.5)

def route_post_to_bots(post_content: str, threshold: float = routing_threshold):
    results = vectorstore.similarity_search_with_relevance_scores(post_content, k=3)
    
    routed_bots = []
    for doc, score in results:
        if score is None:
             score = 0.0
        similarity_score = score
        if similarity_score >= threshold:
            bot_id = doc.metadata.get("id")
            bot_name = doc.metadata.get("name")
            routed_bots.append((bot_id, bot_name, similarity_score))
    
    return routed_bots


In [51]:
post ="Elon Musk plans to integrate AI into financial trading platforms."


In [53]:
route_post_to_bots(post,0.2)

[('bot_a', 'Tech Maximalist', 0.5014173649257084),
 ('bot_c', 'Finance Bro', 0.5006599017642706),
 ('bot_c', 'Finance Bro', 0.48504879336953244)]